# Loss & Class Imbalance Ablation Study
## Independent Validation on RRSIS-D (Binary Segmentation)

**Purpose**: Test whether Custom Segmentation Loss and Class Imbalance strategies
actually help before integrating them into the full RMSIN pipeline.

**Method**: Train a lightweight UNet (∼2M params) on RRSIS-D binary masks with 4 configurations:
- **A) Baseline** — Plain CE + Dice
- **B) Custom Loss** — Focal + Improved Dice + Boundary Loss
- **C) Imbalance-Aware** — OHEM + Dice with dynamic pos_weight
- **D) Combined** — Custom Loss + Imbalance strategies together

**Hardware**: NVIDIA RTX 4050 6GB VRAM (img_size=256, batch_size=8, AMP enabled)

---
## 1. Setup & Installation

In [2]:
!pip install -q einops tabulate


[notice] A new release of pip available: 22.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# ============================================================
# 1a. Environment detection & path setup
# ============================================================
import sys, os

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    DATASET_ROOT = '/content/drive/MyDrive/CSE465/RRSIS-D'
    OUTPUT_DIR   = '/content/drive/MyDrive/CSE465/checkpoints/ablation'
else:
    DATASET_ROOT = './RRSIS-D'
    OUTPUT_DIR   = './checkpoints/ablation'

os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE_DIR = os.path.join(DATASET_ROOT, 'JPEGImages')
LABEL_DIR = os.path.join(DATASET_ROOT, 'ann_split')

print(f'Dataset  : {DATASET_ROOT}')
print(f'Output   : {OUTPUT_DIR}')
print(f'Images OK: {os.path.isdir(IMAGE_DIR)}')
print(f'Labels OK: {os.path.isdir(LABEL_DIR)}')



convert this for my local drive

OUTPUT_DIR   = 'E:\Workplaces\CSE465 Project\Model_Checkpoints\ablation'
DATASET_ROOT = 'E:\Workplaces\CSE465 Project\RRSIS-D'

Dataset  : E:\Workplaces\CSE465 Project\RRSIS-D
Output   : E:\Workplaces\CSE465 Project\Model_Checkpoints\ablation
Images OK: True
Labels OK: True


In [ ]:
# ============================================================
# 1b. Core Imports
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import numpy as np
import random
import json
import pickle
import time
import datetime
import gc
import math
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter
from tqdm.auto import tqdm
from tabulate import tabulate

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Device: cpu
Running on CPU ⚠️ (training will be slow)


c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ============================================================
# 1c. Reproducibility
# ============================================================
def seed_everything(seed=2401):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(2401)
print('Seed set: 2401')

---
## 2. Configuration

In [ ]:
# ============================================================
# 2. Ablation Config — VRAM-friendly for RTX 4050 6GB
# ============================================================
class AblationConfig:
    dataset_root = DATASET_ROOT
    output_dir   = OUTPUT_DIR

    # ── VRAM-friendly settings ──
    img_size     = 256       # smaller than RMSIN's 480
    batch_size   = 8         # fits in 6GB with AMP
    epochs       = 20
    lr           = 1e-3
    weight_decay = 1e-4
    use_amp      = True      # mixed precision

    # DataLoader
    num_workers  = 2
    pin_memory   = True

    # Early stopping
    patience     = 7

    # Model
    unet_base_ch = 32        # SmallUNet base channels (~2M params)

    # Logging
    print_freq   = 10

cfg = AblationConfig()
print(f'img_size={cfg.img_size}, batch_size={cfg.batch_size}, epochs={cfg.epochs}')
print(f'AMP={cfg.use_amp}, UNet base_ch={cfg.unet_base_ch}')

---
## 3. Class Imbalance Analysis

Parse XML annotations from RRSIS-D to understand the category distribution
and compute foreground/background pixel ratios from actual masks.

In [ ]:
# ============================================================
# 3a. Category distribution from XML annotations
# ============================================================
import xml.etree.ElementTree as ET

labels_path = Path(LABEL_DIR)
xml_files = list(labels_path.rglob('*.xml'))
print(f'Total XML annotation files: {len(xml_files)}')

class_names_set = set()
box_counts = Counter()

for xf in xml_files:
    tree = ET.parse(xf)
    root = tree.getroot()
    for obj in root.findall('object'):
        class_name = obj.find('name').text.strip()
        class_names_set.add(class_name)
        box_counts[class_name] += 1

class_names = sorted(list(class_names_set))
num_classes = len(class_names)
class_to_id = {name: i for i, name in enumerate(class_names)}
counts = np.zeros(num_classes, dtype=int)
for name, count in box_counts.items():
    counts[class_to_id[name]] = count

total_boxes = counts.sum()
nonzero = counts[counts > 0]
max_boxes = int(nonzero.max()) if len(nonzero) else 0
min_boxes = int(nonzero.min()) if len(nonzero) else 0
imbalance_ratio = (max_boxes / min_boxes) if min_boxes > 0 else float('inf')
percent_share = (counts / total_boxes * 100) if total_boxes > 0 else np.zeros(num_classes)

print(f'\nCategories: {num_classes}')
print(f'Total objects: {total_boxes}')
print(f'Max: {max_boxes}, Min: {min_boxes}')
print(f'Imbalance ratio: {imbalance_ratio:.2f}x')

print('\nClass-wise distribution:')
print('-' * 60)
print(f'{"ID":>3} {"Class":25s} {"Count":>8} {"Share (%)":>10}')
print('-' * 60)
for i in range(num_classes):
    print(f'{i:3d} {class_names[i]:25s} {counts[i]:8d} {percent_share[i]:10.2f}')

In [ ]:
# ============================================================
# 3b. Imbalance visualization
# ============================================================
idx = np.argsort(-counts)
labels_sorted = [class_names[i] for i in idx]
counts_sorted = counts[idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
ax = axes[0]
bars = ax.barh(range(len(labels_sorted)), counts_sorted, color='steelblue')
ax.set_yticks(range(len(labels_sorted)))
ax.set_yticklabels(labels_sorted, fontsize=8)
ax.set_xlabel('Number of Instances')
ax.set_title('RRSIS-D Category Distribution')
ax.invert_yaxis()

# Pie chart (group small categories)
ax = axes[1]
min_pct = 1.0
big_idx = np.where(percent_share >= min_pct)[0]
small_idx = np.where(percent_share < min_pct)[0]
pie_labels = [class_names[i] for i in big_idx]
pie_sizes = percent_share[big_idx].tolist()
other_pct = float(percent_share[small_idx].sum())
if other_pct > 0:
    pie_labels.append('Other')
    pie_sizes.append(other_pct)
ax.pie(pie_sizes, labels=pie_labels, autopct='%1.1f%%', startangle=90, textprops={'fontsize': 7})
ax.set_title('Category Share (grouped < 1%)')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Class distribution saved.')

In [ ]:
# ============================================================
# 3c. Foreground/Background pixel ratio from actual masks
# ============================================================
instances_path = os.path.join(cfg.dataset_root, 'instances.json')
with open(instances_path, 'r') as f:
    instances = json.load(f)

img_dict = {img['id']: img for img in instances['images']}
ann_dict = {ann['id']: ann for ann in instances['annotations']}

def decode_mask(ann, img_info):
    h, w = img_info['height'], img_info['width']
    if 'segmentation' in ann:
        mask = np.zeros((h, w), dtype=np.uint8)
        seg = ann['segmentation']
        if isinstance(seg, list):
            for poly in seg:
                pts = np.array(poly, dtype=np.float32).reshape(-1, 2)
                pts = pts.astype(np.int32)
                cv2.fillPoly(mask, [pts], 1)
        elif isinstance(seg, dict):
            from pycocotools import mask as coco_mask
            rle = coco_mask.frPyObjects(seg, h, w)
            mask = coco_mask.decode(rle)
        return mask
    return np.zeros((h, w), dtype=np.uint8)

# Sample 300 annotations for speed
sample_anns = random.sample(list(ann_dict.values()), min(300, len(ann_dict)))
fg_pixels, bg_pixels = 0, 0
for ann in tqdm(sample_anns, desc='Computing pixel ratios'):
    img_info = img_dict[ann['image_id']]
    mask = decode_mask(ann, img_info)
    fg = mask.sum()
    total = mask.shape[0] * mask.shape[1]
    fg_pixels += fg
    bg_pixels += (total - fg)

fg_ratio = fg_pixels / (fg_pixels + bg_pixels)
bg_ratio = bg_pixels / (fg_pixels + bg_pixels)
pos_weight = min(bg_pixels / max(fg_pixels, 1), 20.0)  # clamped

print(f'\nForeground pixel ratio: {fg_ratio*100:.2f}%')
print(f'Background pixel ratio: {bg_ratio*100:.2f}%')
print(f'BG/FG ratio: {bg_ratio/max(fg_ratio, 1e-8):.1f}x')
print(f'Computed pos_weight (clamped to 20): {pos_weight:.2f}')

---
## 4. Dataset & Transforms

Uses RRSIS-D as source but builds a **binary segmentation dataset** (no language branch).
Deduplicates by `ann_id` so multiple sentences per reference don't multiply samples.

In [ ]:
# ============================================================
# 4a. Image Transforms (joint image + mask)
# ============================================================
class Compose:
    def __init__(self, transforms):
        self.transforms = transforms
    def __call__(self, image, target):
        for t in self.transforms:
            image, target = t(image, target)
        return image, target

class Resize:
    def __init__(self, h, w):
        self.h, self.w = h, w
    def __call__(self, image, target):
        image = cv2.resize(image, (self.w, self.h), interpolation=cv2.INTER_LINEAR)
        target = cv2.resize(target, (self.w, self.h), interpolation=cv2.INTER_NEAREST)
        return image, target

class RandomHorizontalFlip:
    def __init__(self, p=0.5):
        self.p = p
    def __call__(self, image, target):
        if random.random() < self.p:
            image = np.fliplr(image).copy()
            target = np.fliplr(target).copy()
        return image, target

class ToTensorPair:
    def __call__(self, image, target):
        image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
        target = torch.from_numpy(target).long()
        return image, target

class NormalizePair:
    def __init__(self, mean, std):
        self.mean = torch.tensor(mean).view(3, 1, 1)
        self.std  = torch.tensor(std).view(3, 1, 1)
    def __call__(self, image, target):
        return (image - self.mean) / self.std, target

def get_transform(split='train', img_size=256):
    t = [Resize(img_size, img_size)]
    if split == 'train':
        t.append(RandomHorizontalFlip(0.5))
    t += [ToTensorPair(), NormalizePair([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]
    return Compose(t)

print('Transforms defined.')

In [ ]:
# ============================================================
# 4b. BinarySegDataset — no language branch, deduplicated by ann_id
# ============================================================
class BinarySegDataset(Dataset):
    """
    Binary segmentation dataset from RRSIS-D.
    Each sample = (image, mask) — no text encoder needed.
    Deduplicates by ann_id so each unique mask appears once.
    """
    def __init__(self, cfg, split='train', transform=None):
        self.cfg = cfg
        self.transform = transform
        self.img_dir = os.path.join(cfg.dataset_root, 'JPEGImages')

        with open(os.path.join(cfg.dataset_root, 'instances.json')) as f:
            inst = json.load(f)
        self.img_dict = {img['id']: img for img in inst['images']}
        self.ann_dict = {ann['id']: ann for ann in inst['annotations']}

        with open(os.path.join(cfg.dataset_root, f'refs(unc).p'), 'rb') as f:
            refs = pickle.load(f)

        # Deduplicate: one entry per unique ann_id
        seen_anns = set()
        self.samples = []
        for ref in refs:
            if ref['split'] == split and ref['ann_id'] not in seen_anns:
                seen_anns.add(ref['ann_id'])
                self.samples.append({
                    'ann_id': ref['ann_id'],
                    'image_id': ref['image_id'],
                })
        print(f'[{split}] {len(self.samples)} unique samples (deduplicated by ann_id)')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img_info = self.img_dict[s['image_id']]
        img = cv2.imread(os.path.join(self.img_dir, img_info['file_name']))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = decode_mask(self.ann_dict[s['ann_id']], img_info)

        if self.transform:
            img, mask = self.transform(img, mask)
            mask = mask.long()
        return img, mask

In [ ]:
# ============================================================
# 4c. Build DataLoaders
# ============================================================
train_ds = BinarySegDataset(cfg, 'train', get_transform('train', cfg.img_size))
val_ds   = BinarySegDataset(cfg, 'val',   get_transform('val',   cfg.img_size))
test_ds  = BinarySegDataset(cfg, 'test',  get_transform('val',   cfg.img_size))

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=cfg.num_workers, pin_memory=cfg.pin_memory, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False,
                          num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)

print(f'Train: {len(train_loader)} batches (bs={cfg.batch_size})')
print(f'Val:   {len(val_loader)} batches')
print(f'Test:  {len(test_loader)} batches')

---
## 5. SmallUNet — Lightweight Segmentation Model

A simple encoder-decoder UNet (∼2M params) that fits in RTX 4050 6GB.
This is NOT the RMSIN model — just a proxy to validate loss/imbalance strategies.

In [ ]:
# ============================================================
# 5. SmallUNet (~2M params, base_ch=32)
# ============================================================
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class SmallUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=2, base=32):
        super().__init__()
        # Encoder
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.enc4 = ConvBlock(base * 4, base * 8)
        # Bottleneck
        self.bottleneck = ConvBlock(base * 8, base * 16)
        # Decoder
        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = ConvBlock(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)
        # Head
        self.head = nn.Conv2d(base, out_ch, 1)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        # Bottleneck
        b = self.bottleneck(self.pool(e4))
        # Decoder with skip connections
        d4 = self.dec4(torch.cat([self.up4(b), e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.head(d1)

# Quick check
_m = SmallUNet(base=cfg.unet_base_ch)
_p = sum(p.numel() for p in _m.parameters())
print(f'SmallUNet: {_p:,} params ({_p/1e6:.2f}M)')
del _m

---
## 6. Loss Functions

Four loss configurations to compare:
1. **PlainCEDiceLoss** — Baseline (CE + Dice)
2. **CompositeLoss** — Focal + Improved Dice + Boundary
3. **ImbalanceAwareCompound** — OHEM + Dice with pos_weight
4. **CompositeWithImbalance** — Composite + λ·OHEM

In [ ]:
# ============================================================
# 6a. Loss A: PlainCEDiceLoss (Baseline)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        C = pred.shape[1]
        pred_soft = F.softmax(pred, dim=1)
        target_oh = torch.stack([target == c for c in range(C)], dim=1).float()
        inter = (pred_soft * target_oh).sum(dim=(2, 3))
        union = (pred_soft + target_oh).sum(dim=(2, 3))
        return (1 - (2 * inter + self.smooth) / (union + self.smooth)).mean()

class PlainCEDiceLoss(nn.Module):
    """Baseline: 0.9 * CE + 0.1 * Dice (matches original notebook)."""
    def __init__(self, ce_weight=None, dice_weight=0.1):
        super().__init__()
        self.dice_weight = dice_weight
        self.ce_weight = ce_weight  # class weights tensor or None
        self.dice = DiceLoss()

    def forward(self, pred, target):
        if pred.shape[-2:] != target.shape[-2:]:
            pred = F.interpolate(pred, size=target.shape[-2:],
                                 mode='bilinear', align_corners=False)
        target_long = target.squeeze(1).long()
        ce = F.cross_entropy(pred, target_long, weight=self.ce_weight)
        dice = self.dice(pred, target_long)
        return (1 - self.dice_weight) * ce + self.dice_weight * dice

print('PlainCEDiceLoss defined (baseline).')

In [ ]:
# ============================================================
# 6b. Loss B: CompositeLoss (Focal + Improved Dice + Boundary)
# ============================================================
class ImprovedDiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):
        pred_soft = F.softmax(pred, dim=1)[:, 1]  # foreground prob
        target_f = target.float()
        inter = (pred_soft * target_f).sum(dim=(1, 2))
        union = pred_soft.sum(dim=(1, 2)) + target_f.sum(dim=(1, 2))
        dice = (2 * inter + self.smooth) / (union + self.smooth)
        return (1 - dice).mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        if alpha is not None:
            self.register_buffer('alpha', torch.FloatTensor(alpha))
        else:
            self.alpha = None

    def forward(self, pred, target):
        ce = F.cross_entropy(pred, target, reduction='none')
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        if self.alpha is not None:
            alpha_t = self.alpha[target]
            focal = alpha_t * focal
        return focal.mean()

class BoundaryLoss(nn.Module):
    def __init__(self, theta=3):
        super().__init__()
        self.theta = theta
        kernel = torch.tensor([[0, 1, 0],
                               [1, -4, 1],
                               [0, 1, 0]], dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        self.register_buffer('laplacian', kernel)

    def get_boundary(self, mask):
        mask_f = mask.float().unsqueeze(1)
        edges = F.conv2d(mask_f, self.laplacian, padding=1)
        edges = (edges.abs() > 0).float().squeeze(1)
        if self.theta > 1:
            kernel_size = 2 * self.theta + 1
            edges = F.max_pool2d(edges.unsqueeze(1), kernel_size,
                                 stride=1, padding=self.theta).squeeze(1)
        return edges

    def forward(self, pred, target):
        boundary = self.get_boundary(target)
        pred_soft = F.softmax(pred, dim=1)[:, 1]
        boundary_pred = pred_soft * boundary
        boundary_gt = target.float() * boundary
        num_boundary = boundary.sum().clamp(min=1)
        loss = F.binary_cross_entropy(
            boundary_pred.clamp(1e-6, 1 - 1e-6),
            boundary_gt, reduction='sum'
        ) / num_boundary
        return loss

class CompositeLoss(nn.Module):
    """Focal(alpha=[0.6,1.4], gamma=2) + ImprovedDice + BoundaryLoss."""
    def __init__(self, w_focal=0.5, w_dice=0.3, w_boundary=0.2):
        super().__init__()
        self.w_focal = w_focal
        self.w_dice = w_dice
        self.w_boundary = w_boundary
        self.focal = FocalLoss(alpha=[0.6, 1.4], gamma=2.0)
        self.dice = ImprovedDiceLoss()
        self.boundary = BoundaryLoss(theta=3)

    def forward(self, pred, target):
        if pred.shape[-2:] != target.shape[-2:]:
            pred = F.interpolate(pred, size=target.shape[-2:],
                                 mode='bilinear', align_corners=False)
        target_long = target.squeeze(1).long()
        L_focal = self.focal(pred, target_long)
        L_dice  = self.dice(pred, target_long)
        L_bnd   = self.boundary(pred, target_long)
        return self.w_focal * L_focal + self.w_dice * L_dice + self.w_boundary * L_bnd

print('CompositeLoss defined (Focal + Dice + Boundary).')

In [ ]:
# ============================================================
# 6c. Loss C: ImbalanceAwareCompound (OHEM + Dice + pos_weight)
# ============================================================
class OHEMLoss(nn.Module):
    """Online Hard Example Mining: keep top hard_ratio of pixels."""
    def __init__(self, hard_ratio=0.3, pos_weight=None):
        super().__init__()
        self.hard_ratio = hard_ratio
        if pos_weight is not None:
            self.register_buffer('pw', torch.FloatTensor([1.0, pos_weight]))
        else:
            self.pw = None

    def forward(self, pred, target):
        weight = self.pw if self.pw is not None else None
        ce = F.cross_entropy(pred, target, weight=weight, reduction='none')
        B = ce.size(0)
        ce_flat = ce.view(B, -1)
        k = max(1, int(ce_flat.size(1) * self.hard_ratio))
        topk_loss, _ = ce_flat.topk(k, dim=1)
        return topk_loss.mean()

class ImbalanceAwareCompound(nn.Module):
    """OHEM(pos_weight) + Dice."""
    def __init__(self, pos_weight=None, hard_ratio=0.3, dice_weight=0.3):
        super().__init__()
        self.dice_weight = dice_weight
        # Use computed pos_weight from Section 3c, default to 10.0
        pw = pos_weight if pos_weight is not None else globals().get('pos_weight', 10.0)
        self.ohem = OHEMLoss(hard_ratio=hard_ratio, pos_weight=pw)
        self.dice = DiceLoss()

    def forward(self, pred, target):
        if pred.shape[-2:] != target.shape[-2:]:
            pred = F.interpolate(pred, size=target.shape[-2:],
                                 mode='bilinear', align_corners=False)
        target_long = target.squeeze(1).long()
        L_ohem = self.ohem(pred, target_long)
        L_dice = self.dice(pred, target_long)
        return (1 - self.dice_weight) * L_ohem + self.dice_weight * L_dice

print('ImbalanceAwareCompound defined (OHEM + Dice with dynamic pos_weight).')

In [ ]:
# ============================================================
# 6d. Loss D: CompositeWithImbalance (Composite + lambda*OHEM)
# ============================================================
class CompositeWithImbalance(nn.Module):
    """CompositeLoss + lambda * OHEM for the full combination."""
    def __init__(self, lam=0.3, pos_weight=None):
        super().__init__()
        self.lam = lam
        self.composite = CompositeLoss()
        pw = pos_weight if pos_weight is not None else globals().get('pos_weight', 10.0)
        self.ohem = OHEMLoss(hard_ratio=0.3, pos_weight=pw)

    def forward(self, pred, target):
        if pred.shape[-2:] != target.shape[-2:]:
            pred = F.interpolate(pred, size=target.shape[-2:],
                                 mode='bilinear', align_corners=False)
        target_long = target.squeeze(1).long()
        L_comp = self.composite(pred, target_long)
        L_ohem = self.ohem(pred, target_long)
        return L_comp + self.lam * L_ohem

print('CompositeWithImbalance defined (Composite + 0.3*OHEM).')

---
## 7. Evaluation Metrics

In [ ]:
# ============================================================
# 7. Segmentation Metrics (oIoU, mIoU, Pr@K, F1, BoundaryIoU)
# ============================================================
class SegmentationMetrics:
    THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9]

    def __init__(self):
        self.reset()

    def reset(self):
        self._total_inter = 0.0
        self._total_union = 0.0
        self._per_image_ious = []
        self._tp = 0; self._fp = 0; self._fn = 0  # for F1
        self._boundary_ious = []

    def _get_boundary(self, mask, thickness=2):
        """Extract boundary pixels from binary mask."""
        kernel = torch.ones(1, 1, 2 * thickness + 1, 2 * thickness + 1,
                           device=mask.device)
        dilated = F.conv2d(mask.float().unsqueeze(0).unsqueeze(0),
                          kernel, padding=thickness).squeeze() > 0
        eroded = F.conv2d(mask.float().unsqueeze(0).unsqueeze(0),
                         kernel, padding=thickness).squeeze() >= kernel.numel()
        return (dilated.float() - eroded.float()).clamp(0, 1)

    def update(self, pred_mask, gt_mask):
        pred = (pred_mask > 0.5).float().view(pred_mask.shape[0], -1)
        gt   = (gt_mask   > 0.5).float().view(gt_mask.shape[0], -1)
        inter = (pred * gt).sum(dim=1)
        union = (pred + gt).clamp(max=1).sum(dim=1)

        self._total_inter += inter.sum().item()
        self._total_union += union.sum().item()

        per_iou = torch.where(union > 0, inter / union, torch.ones_like(inter))
        self._per_image_ious.extend(per_iou.cpu().tolist())

        # F1 components
        self._tp += inter.sum().item()
        self._fp += (pred * (1 - gt)).sum().item()
        self._fn += ((1 - pred) * gt).sum().item()

        # Boundary IoU (per-image)
        for b in range(pred_mask.shape[0]):
            p = (pred_mask[b].squeeze() > 0.5)
            g = (gt_mask[b].squeeze() > 0.5)
            if g.sum() == 0 and p.sum() == 0:
                self._boundary_ious.append(1.0)
                continue
            if g.sum() == 0 or p.sum() == 0:
                self._boundary_ious.append(0.0)
                continue
            pb = self._get_boundary(p, 2)
            gb = self._get_boundary(g, 2)
            bi = (pb * gb).sum().item()
            bu = (pb + gb).clamp(max=1).sum().item()
            self._boundary_ious.append(bi / max(bu, 1e-8))

    def compute(self):
        oiou = self._total_inter / (self._total_union + 1e-8)
        ious  = np.array(self._per_image_ious)
        miou  = float(ious.mean())
        prec = self._tp / (self._tp + self._fp + 1e-8)
        rec  = self._tp / (self._tp + self._fn + 1e-8)
        f1 = 2 * prec * rec / (prec + rec + 1e-8)
        biou = float(np.mean(self._boundary_ious)) if self._boundary_ious else 0.0
        results = {"oIoU": oiou, "mIoU": miou, "F1": f1, "BIoU": biou}
        for k in self.THRESHOLDS:
            results[f"Pr@{k}"] = float((ious >= k).mean())
        return results

print('SegmentationMetrics defined (oIoU, mIoU, F1, BIoU, Pr@K).')

In [ ]:
# ============================================================
# 7b. RunLogger — per-run CSV logging
# ============================================================
class RunLogger:
    def __init__(self, run_name, save_dir):
        self.run_name = run_name
        self.save_dir = Path(save_dir)
        self.save_dir.mkdir(parents=True, exist_ok=True)
        self.history = defaultdict(list)

    def log(self, epoch, **kwargs):
        self.history['epoch'].append(epoch)
        for k, v in kwargs.items():
            self.history[k].append(float(v))

    def save_csv(self):
        import csv
        path = self.save_dir / f'results_{self.run_name}.csv'
        keys = list(self.history.keys())
        with open(path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            for i in range(len(self.history['epoch'])):
                writer.writerow({k: self.history[k][i] for k in keys})
        print(f'  Results saved: {path}')
        return path

print('RunLogger defined.')

---
## 8. Training Infrastructure

In [ ]:
# ============================================================
# 8a. Train one epoch
# ============================================================
def train_one_epoch(model, loader, criterion, optimizer, scheduler, epoch, cfg, metrics):
    model.train()
    metrics.reset()
    total_loss = 0.0
    N = len(loader)
    scaler = torch.amp.GradScaler('cuda', enabled=cfg.use_amp)

    pbar = tqdm(loader, desc=f'Epoch {epoch} [Train]', leave=False)
    for i, (imgs, masks) in enumerate(pbar):
        imgs  = imgs.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', enabled=cfg.use_amp):
            output = model(imgs)
            loss = criterion(output, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()
        with torch.no_grad():
            pred_up = F.interpolate(output.detach(), size=masks.shape[-2:],
                                    mode='bilinear', align_corners=False)
            pred_prob = torch.softmax(pred_up, dim=1)[:, 1:, :, :]
            metrics.update(pred_prob, masks)

        pbar.set_postfix(loss=f'{loss.item():.4f}', avg=f'{total_loss/(i+1):.4f}')
        del imgs, masks, output, loss

    return total_loss / N, metrics.compute()

# ============================================================
# 8b. Evaluate
# ============================================================
@torch.no_grad()
def evaluate(model, loader, criterion, epoch, cfg, metrics):
    model.eval()
    metrics.reset()
    total_loss = 0.0
    n = 0
    pbar = tqdm(loader, desc=f'Epoch {epoch} [Val]', leave=False)
    for imgs, masks in pbar:
        imgs  = imgs.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        with torch.autocast('cuda', enabled=cfg.use_amp):
            out = model(imgs)
            loss = criterion(out, masks)
        total_loss += loss.item()
        n += 1
        pred_up = F.interpolate(out, size=masks.shape[-2:],
                                mode='bilinear', align_corners=False)
        pred_prob = torch.softmax(pred_up, dim=1)[:, 1:, :, :]
        metrics.update(pred_prob, masks)
        pbar.set_postfix(loss=f'{total_loss/n:.4f}')
    return total_loss / max(n, 1), metrics.compute()

print('Training functions defined.')

In [ ]:
# ============================================================
# 8c. run_experiment() — trains a fresh model with given criterion
# ============================================================
def run_experiment(run_name, criterion, cfg):
    print(f'\n{"="*60}')
    print(f'  EXPERIMENT: {run_name}')
    print(f'{"="*60}')

    seed_everything(2401)  # same seed for fair comparison
    model = SmallUNet(base=cfg.unet_base_ch).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr,
                                   weight_decay=cfg.weight_decay)
    total_steps = len(train_loader) * cfg.epochs
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_steps, eta_min=cfg.lr * 0.01)

    criterion = criterion.to(DEVICE)
    train_metrics = SegmentationMetrics()
    val_metrics   = SegmentationMetrics()
    logger = RunLogger(run_name, cfg.output_dir)

    best_oiou = -1.0
    best_epoch = 0
    patience_counter = 0

    for epoch in range(cfg.epochs):
        train_loss, train_stats = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, epoch, cfg, train_metrics)
        val_loss, val_stats = evaluate(
            model, val_loader, criterion, epoch, cfg, val_metrics)

        logger.log(epoch,
                   train_loss=train_loss, val_loss=val_loss,
                   train_oIoU=train_stats['oIoU'], val_oIoU=val_stats['oIoU'],
                   train_mIoU=train_stats['mIoU'], val_mIoU=val_stats['mIoU'],
                   val_F1=val_stats['F1'], val_BIoU=val_stats['BIoU'],
                   **{f'val_Pr@{k}': v for k, v in val_stats.items() if 'Pr@' in str(k)})

        is_best = val_stats['oIoU'] > best_oiou
        if is_best:
            best_oiou = val_stats['oIoU']
            best_epoch = epoch
            patience_counter = 0
            torch.save(model.state_dict(),
                       os.path.join(cfg.output_dir, f'best_{run_name}.pth'))
        else:
            patience_counter += 1

        print(f'  Ep {epoch:2d} | TrL={train_loss:.4f} VL={val_loss:.4f} | '
              f'oIoU={val_stats["oIoU"]*100:.2f}% mIoU={val_stats["mIoU"]*100:.2f}% '
              f'F1={val_stats["F1"]*100:.2f}% BIoU={val_stats["BIoU"]*100:.2f}% | '
              f'Best={best_oiou*100:.2f}% pat={patience_counter}/{cfg.patience}')

        if patience_counter >= cfg.patience:
            print(f'  Early stopping at epoch {epoch}.')
            break

        gc.collect()
        torch.cuda.empty_cache()

    logger.save_csv()
    print(f'  >> Best oIoU: {best_oiou*100:.2f}% at epoch {best_epoch}')

    del model, optimizer, scheduler
    gc.collect()
    torch.cuda.empty_cache()

    return best_oiou, best_epoch, logger

print('run_experiment() defined.')

---
## 9. Run All 4 Experiments

In [ ]:
# ============================================================
# 9a. Run A — Baseline (Plain CE + Dice)
# ============================================================
best_A, epoch_A, log_A = run_experiment(
    'A_baseline',
    PlainCEDiceLoss(dice_weight=0.1),
    cfg
)

In [ ]:
# ============================================================
# 9b. Run B — Custom Loss (Focal + Dice + Boundary)
# ============================================================
best_B, epoch_B, log_B = run_experiment(
    'B_custom_loss',
    CompositeLoss(),
    cfg
)

In [ ]:
# ============================================================
# 9c. Run C — Imbalance-Aware (OHEM + Dice + pos_weight)
# ============================================================
best_C, epoch_C, log_C = run_experiment(
    'C_imbalance',
    ImbalanceAwareCompound(),
    cfg
)

In [ ]:
# ============================================================
# 9d. Run D — Combined (Composite + OHEM)
# ============================================================
best_D, epoch_D, log_D = run_experiment(
    'D_combined',
    CompositeWithImbalance(),
    cfg
)

---
## 10. Side-by-Side Comparison

In [ ]:
# ============================================================
# 10. Side-by-side comparison plots
# ============================================================
import csv

def load_results_csv(path):
    data = defaultdict(list)
    with open(path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            for k, v in row.items():
                data[k].append(float(v))
    return data

runs = {
    'A_baseline':    ('Baseline (CE+Dice)', '#2196F3'),
    'B_custom_loss': ('Custom Loss',        '#F44336'),
    'C_imbalance':   ('Imbalance-Aware',    '#4CAF50'),
    'D_combined':    ('Combined',            '#FF9800'),
}

all_data = {}
for run_name in runs:
    csv_path = os.path.join(cfg.output_dir, f'results_{run_name}.csv')
    if os.path.exists(csv_path):
        all_data[run_name] = load_results_csv(csv_path)

metrics_to_plot = [
    ('val_loss',  'Val Loss',  False),
    ('val_oIoU',  'Val oIoU',  True),
    ('val_mIoU',  'Val mIoU',  True),
    ('val_F1',    'Val F1',    True),
    ('val_BIoU',  'Val BIoU',  True),
    ('val_Pr@0.5','Val Pr@0.5',True),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Ablation Study: Loss & Imbalance Strategies', fontsize=14, fontweight='bold')

for ax, (key, title, higher_better) in zip(axes.flat, metrics_to_plot):
    for run_name, (label, color) in runs.items():
        if run_name in all_data and key in all_data[run_name]:
            epochs = all_data[run_name]['epoch']
            values = all_data[run_name][key]
            ax.plot(epochs, values, label=label, color=color, linewidth=2, marker='o', markersize=3)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, 'ablation_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Comparison plot saved.')

In [ ]:
# ============================================================
# 10b. Best-epoch summary table + delta analysis
# ============================================================
summary_rows = []
baseline_best = None

for run_name, (label, color) in runs.items():
    if run_name not in all_data:
        continue
    d = all_data[run_name]
    # Find best epoch by val_oIoU
    best_idx = int(np.argmax(d['val_oIoU']))
    row = {
        'Run': label,
        'Best Epoch': int(d['epoch'][best_idx]),
        'Val Loss': d['val_loss'][best_idx],
        'oIoU': d['val_oIoU'][best_idx] * 100,
        'mIoU': d['val_mIoU'][best_idx] * 100,
        'F1':   d['val_F1'][best_idx] * 100,
        'BIoU': d['val_BIoU'][best_idx] * 100,
        'Pr@0.5': d['val_Pr@0.5'][best_idx] * 100,
    }
    if run_name == 'A_baseline':
        baseline_best = row
    summary_rows.append(row)

# Print table
print('\n' + '=' * 90)
print('  BEST-EPOCH SUMMARY')
print('=' * 90)
headers = ['Run', 'Best Epoch', 'Val Loss', 'oIoU', 'mIoU', 'F1', 'BIoU', 'Pr@0.5']
table_data = [[r[h] if isinstance(r[h], str) else f'{r[h]:.2f}' for h in headers] for r in summary_rows]
print(tabulate(table_data, headers=headers, tablefmt='rounded_outline'))

# Delta analysis
if baseline_best:
    print('\n' + '=' * 90)
    print('  DELTA vs BASELINE')
    print('=' * 90)
    delta_rows = []
    for row in summary_rows:
        if row['Run'] == baseline_best['Run']:
            continue
        deltas = {}
        for metric in ['oIoU', 'mIoU', 'F1', 'BIoU', 'Pr@0.5']:
            d = row[metric] - baseline_best[metric]
            verdict = 'HELPS' if d >= 1.0 else ('HURTS' if d <= -1.0 else 'NEUTRAL')
            deltas[metric] = f'{d:+.2f}pp ({verdict})'
        delta_rows.append([row['Run']] + [deltas[m] for m in ['oIoU', 'mIoU', 'F1', 'BIoU', 'Pr@0.5']])
    print(tabulate(delta_rows,
                   headers=['Run', 'Δ oIoU', 'Δ mIoU', 'Δ F1', 'Δ BIoU', 'Δ Pr@0.5'],
                   tablefmt='rounded_outline'))

---
## 11. Test Set Evaluation

In [ ]:
# ============================================================
# 11. Test evaluation — reload each best checkpoint
# ============================================================
test_results = {}

for run_name, (label, color) in runs.items():
    ckpt_path = os.path.join(cfg.output_dir, f'best_{run_name}.pth')
    if not os.path.exists(ckpt_path):
        print(f'  [skip] {run_name}: no checkpoint found')
        continue

    seed_everything(2401)
    model = SmallUNet(base=cfg.unet_base_ch).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()

    test_metrics = SegmentationMetrics()
    # Use PlainCEDiceLoss for consistent test loss measurement
    test_criterion = PlainCEDiceLoss().to(DEVICE)
    test_loss, test_stats = evaluate(model, test_loader, test_criterion, 0, cfg, test_metrics)
    test_results[run_name] = {'loss': test_loss, **test_stats}

    print(f'  {label:25s} | oIoU={test_stats["oIoU"]*100:.2f}% '
          f'mIoU={test_stats["mIoU"]*100:.2f}% F1={test_stats["F1"]*100:.2f}% '
          f'BIoU={test_stats["BIoU"]*100:.2f}%')
    del model
    gc.collect()
    torch.cuda.empty_cache()

# Save test results
test_results_path = os.path.join(cfg.output_dir, 'test_results.json')
with open(test_results_path, 'w') as f:
    json.dump({k: {kk: float(vv) for kk, vv in v.items()} for k, v in test_results.items()}, f, indent=2)
print(f'\nTest results saved: {test_results_path}')

---
## 12. Decision: Should We Integrate These Into RMSIN?

In [ ]:
# ============================================================
# 12. Decision rule: merge if Δ >= 1pp on oIoU or F1 or BIoU
# ============================================================
if 'A_baseline' in test_results:
    baseline = test_results['A_baseline']
    print('\n' + '=' * 70)
    print('  FINAL DECISION: Integrate into RMSIN pipeline?')
    print('=' * 70)

    for run_name, (label, _) in runs.items():
        if run_name == 'A_baseline' or run_name not in test_results:
            continue
        r = test_results[run_name]
        d_oiou = (r['oIoU'] - baseline['oIoU']) * 100
        d_f1   = (r['F1']   - baseline['F1']) * 100
        d_biou = (r['BIoU'] - baseline['BIoU']) * 100

        should_merge = d_oiou >= 1.0 or d_f1 >= 1.0 or d_biou >= 1.0
        verdict = 'YES — MERGE' if should_merge else 'NO — SKIP'

        print(f'\n  {label}:')
        print(f'    Δ oIoU: {d_oiou:+.2f}pp | Δ F1: {d_f1:+.2f}pp | Δ BIoU: {d_biou:+.2f}pp')
        print(f'    Verdict: {verdict}')

    print('\n' + '=' * 70)
    print('  Threshold: Δ >= 1pp on any of oIoU, F1, or BIoU')
    print('=' * 70)
else:
    print('Run all experiments first before making a decision.')

---
## End of Ablation Study